# Crys-JEPA + DFT Superconductivity Training — Local

Notebook pour entraîner les variantes **DFT seul** et **DFT-JEPA** sur machine locale.

Pré-requis :
- Environnement Python installé via `uv sync` depuis la racine du projet
- GPU optionnel : CUDA (Linux/Windows) ou MPS (Apple Silicon) — CPU en fallback

Configs utilisées :
- `configs/ablation/dft.yaml`
- `configs/ablation/crys_jepa_dft.yaml`

Les checkpoints sont sauvegardés dans `checkpoints/local_<mode>/`.

In [1]:
# Vérifier le GPU disponible
import torch

if torch.cuda.is_available():
    DEVICE = 'cuda'
    print('CUDA:', torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('MPS (Apple Silicon)')
else:
    DEVICE = 'cpu'
    print('CPU only')

print('torch:', torch.__version__)
print('device:', DEVICE)

MPS (Apple Silicon)
torch: 2.6.0
device: mps


In [3]:
# Configuration
from pathlib import Path
import os

# Racine du projet (le notebook est dans notebooks/, on remonte d'un niveau)
PROJECT_DIR = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

os.chdir(PROJECT_DIR)
print('Working directory:', PROJECT_DIR)

# Chemins données
CSV_PATH = PROJECT_DIR / 'data/raw/3DSC_MP.csv'
CIF_DIR  = PROJECT_DIR / 'data/raw/cifs'

# Hyperparamètres (ajuste selon ta machine)
EPOCHS       = 50
BATCH_SIZE   = 32
LEARNING_RATE = 1e-4

Working directory: /Users/baptistecaillerie/Documents/Crys_JEPA


In [4]:
# Vérifier les dépendances minimales
import importlib.util

missing = []
for pkg, import_name in [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('pyyaml', 'yaml'),
    ('easydict', 'easydict'),
    ('tqdm', 'tqdm'),
    ('pytest', 'pytest'),
    ('pymatgen', 'pymatgen'),
    ('sklearn', 'sklearn'),
]:
    if importlib.util.find_spec(import_name) is None:
        missing.append(pkg)

if missing:
    print('Packages manquants:', missing)
    print('Installe via : uv sync  (ou pip install', ' '.join(missing), ')')
else:
    print('Toutes les dépendances sont présentes.')

Toutes les dépendances sont présentes.


In [5]:
# Télécharger le jeu de données 3DSC depuis aimat-lab/3DSC (GitHub)
import urllib.request
import zipfile

n_cifs = len(list(CIF_DIR.glob('*.cif'))) if CIF_DIR.exists() else 0

if CSV_PATH.exists() and n_cifs > 0:
    print(f'Données déjà présentes — CSV: {CSV_PATH}, CIFs: {n_cifs}')
else:
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    CIF_DIR.mkdir(parents=True, exist_ok=True)

    if not CSV_PATH.exists():
        CSV_URL = (
            'https://raw.githubusercontent.com/aimat-lab/3DSC/main/'
            'superconductors_3D/data/final/MP/3DSC_MP.csv'
        )
        print('Téléchargement du CSV...')
        urllib.request.urlretrieve(CSV_URL, CSV_PATH)
        print(f'  CSV: {CSV_PATH} ({CSV_PATH.stat().st_size // 1024} Ko)')
    else:
        print('CSV déjà présent.')

    if n_cifs == 0:
        REPO_ZIP_URL = 'https://github.com/aimat-lab/3DSC/archive/refs/heads/main.zip'
        zip_path = PROJECT_DIR / 'data/raw/3DSC_repo.zip'

        print('Téléchargement du repo aimat-lab/3DSC (~40 Mo)...')
        urllib.request.urlretrieve(REPO_ZIP_URL, zip_path)
        print(f'  Archive: {zip_path.stat().st_size // (1024**2)} Mo')

        CIFS_MARKER = 'superconductors_3D/data/final/MP/cifs/'
        print('Extraction des CIFs...')
        n_extracted = 0
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for entry in zf.namelist():
                if CIFS_MARKER in entry and entry.endswith('.cif'):
                    target = CIF_DIR / Path(entry).name
                    with zf.open(entry) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
                    n_extracted += 1
        zip_path.unlink()
        print(f'  {n_extracted} CIFs extraits dans {CIF_DIR}')
    else:
        print(f'CIFs déjà présents ({n_cifs} fichiers).')

print()
print('CSV exists:', CSV_PATH.exists())
print('CIF count:', len(list(CIF_DIR.glob('*.cif'))))

Téléchargement du CSV...
  CSV: /Users/baptistecaillerie/Documents/Crys_JEPA/data/raw/3DSC_MP.csv (9235 Ko)
Téléchargement du repo aimat-lab/3DSC (~40 Mo)...
  Archive: 17 Mo
Extraction des CIFs...
  10904 CIFs extraits dans /Users/baptistecaillerie/Documents/Crys_JEPA/data/raw/cifs

CSV exists: True
CIF count: 10904


In [6]:
# Patcher les configs pour l'entraînement local
import yaml

CONFIGS = {
    'dft':           PROJECT_DIR / 'configs/ablation/dft.yaml',
    'crys_jepa_dft': PROJECT_DIR / 'configs/ablation/crys_jepa_dft.yaml',
}

for name, cfg_path in CONFIGS.items():
    with cfg_path.open('r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    cfg['training']['epochs']        = int(EPOCHS)
    cfg['training']['batch_size']    = int(BATCH_SIZE)
    cfg['training']['learning_rate'] = float(LEARNING_RATE)
    cfg['training']['device']        = DEVICE
    cfg['checkpoints']['save_dir']   = f'checkpoints/local_{name}'
    with cfg_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print(name, '->', cfg_path)
    print('  save_dir:   ', cfg['checkpoints']['save_dir'])
    print('  input_mode: ', cfg['model']['input_mode'])
    print('  device:     ', cfg['training']['device'])

dft -> /Users/baptistecaillerie/Documents/Crys_JEPA/configs/ablation/dft.yaml
  save_dir:    checkpoints/local_dft
  input_mode:  dft
  device:      mps
crys_jepa_dft -> /Users/baptistecaillerie/Documents/Crys_JEPA/configs/ablation/crys_jepa_dft.yaml
  save_dir:    checkpoints/local_crys_jepa_dft
  input_mode:  crys_jepa_dft
  device:      mps


In [7]:
# Lancer les tests rapides du MVP
!python -m pytest tests/test_superconductivity_mvp.py -q

..............                                                           [100%]
=============================== warnings summary ===============================
tests/test_superconductivity_mvp.py::test_dataset_reads_csv_and_extracts_crystal_tensors
tests/test_superconductivity_mvp.py::test_dataset_reads_csv_and_extracts_crystal_tensors
tests/test_superconductivity_mvp.py::test_split_dataset_is_reproducible
tests/test_superconductivity_mvp.py::test_dataset_ignores_comment_lines_and_resolves_existing_csv_cif_paths
tests/test_superconductivity_mvp.py::test_dataset_ignores_comment_lines_and_resolves_existing_csv_cif_paths
tests/test_superconductivity_mvp.py::test_dataset_falls_back_to_flat_cif_dir_when_csv_contains_historical_path
tests/test_superconductivity_mvp.py::test_dataset_falls_back_to_flat_cif_dir_when_csv_contains_historical_path
tests/test_superconductivity_mvp.py::test_dataset_returns_scaled_dft_features_from_configured_columns
tests/test_superconductivity_mvp.py::test_dataset

In [8]:
# Entraîner DFT seul
!python scripts/train_mvp.py --config configs/ablation/dft.yaml

Training 3DSC MVP:   0%|                                | 0/50 [00:00<?, ?it/s]/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/composition.py:1372: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/composition.py:1372: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatg

In [9]:
# Entraîner DFT-JEPA
!python scripts/train_mvp.py --config configs/ablation/crys_jepa_dft.yaml

Training 3DSC MVP:   0%|                                | 0/50 [00:00<?, ?it/s]/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/composition.py:1372: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/composition.py:1372: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatg

In [10]:
# Évaluer et sauvegarder les prédictions
!python scripts/evaluate_mvp.py \
    --config configs/ablation/dft.yaml \
    --checkpoint checkpoints/local_dft/best.pt \
    --predictions-csv predictions_dft.csv

!python scripts/evaluate_mvp.py \
    --config configs/ablation/crys_jepa_dft.yaml \
    --checkpoint checkpoints/local_crys_jepa_dft/best.pt \
    --predictions-csv predictions_crys_jepa_dft.csv

/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/composition.py:1372: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/composition.py:1372: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 2

In [11]:
# Tableau comparatif depuis les checkpoints
import torch
import pandas as pd

rows = []
for mode, ckpt_path in {
    'dft':           'checkpoints/local_dft/best.pt',
    'crys_jepa_dft': 'checkpoints/local_crys_jepa_dft/best.pt',
}.items():
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    metrics = ckpt.get('val_metrics', {})
    cls = metrics.get('classification', {})
    reg = metrics.get('regression', {})
    rows.append({
        'mode': mode,
        'epoch': ckpt.get('epoch'),
        'val_accuracy': cls.get('accuracy'),
        'val_f1': cls.get('f1'),
        'val_roc_auc': cls.get('roc_auc'),
        'val_mae': reg.get('mae'),
        'val_rmse': reg.get('rmse'),
        'val_mae_superconductors': reg.get('mae_superconductors'),
        'val_mae_high_tc': reg.get('mae_high_tc'),
    })

summary = pd.DataFrame(rows)
summary.to_csv('local_ablation_validation_summary.csv', index=False)
summary

,mode,epoch,val_accuracy,val_f1,val_roc_auc,val_mae,val_rmse,val_mae_superconductors,val_mae_high_tc
0,dft,49,0.684575,0.807611,0.625120,6.908214,13.381177,7.099422,22.235342
1,crys_jepa_dft,50,0.766031,0.840989,0.796723,4.848222,11.224051,4.252607,10.366974


## Phase 3 — Évaluation robuste multi-seeds

**But :** éviter qu'un résultat dépende d'un seul split aléatoire.

- Seeds testées : 0, 1, 2, 3, 4
- `use_amp` activé si CUDA disponible (Tensor Cores)
- Résultat : moyenne ± écart-type → l'amélioration DFT-JEPA est-elle stable ?

In [12]:
# Configuration Phase 3
SEEDS = [0, 1, 2, 3, 4]
EPOCHS_MULTISEED = EPOCHS
MULTISEED_OUTPUT_DIR = 'results/multiseed'

print(f'Seeds : {SEEDS}')
print(f'Époques par run : {EPOCHS_MULTISEED}')
print(f'Total runs : {len(SEEDS) * 2} (dft × {len(SEEDS)} + crys_jepa_dft × {len(SEEDS)})')

Seeds : [0, 1, 2, 3, 4]
Époques par run : 50
Total runs : 10 (dft × 5 + crys_jepa_dft × 5)


In [13]:
# Patcher les configs pour Phase 3
import yaml

CONFIGS_MULTISEED = {
    'dft':           PROJECT_DIR / 'configs/ablation/dft.yaml',
    'crys_jepa_dft': PROJECT_DIR / 'configs/ablation/crys_jepa_dft.yaml',
}

use_amp = DEVICE == 'cuda'  # AMP uniquement sur CUDA

for name, cfg_path in CONFIGS_MULTISEED.items():
    with cfg_path.open('r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    cfg['training']['epochs']        = int(EPOCHS_MULTISEED)
    cfg['training']['batch_size']    = int(BATCH_SIZE)
    cfg['training']['learning_rate'] = float(LEARNING_RATE)
    cfg['training']['device']        = DEVICE
    cfg['training']['use_amp']       = use_amp
    cfg['training']['num_workers']   = 2
    cfg['training']['pin_memory']    = DEVICE == 'cuda'
    cfg['training']['seed']          = 42
    cfg['checkpoints']['save_dir']   = f'checkpoints/local_{name}'
    with cfg_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print(f'✓ {name}: device={DEVICE}, AMP={use_amp}, num_workers=2')

✓ dft: device=mps, AMP=False, num_workers=2
✓ crys_jepa_dft: device=mps, AMP=False, num_workers=2


In [14]:
# Lancer l'ablation multi-seeds (10 runs : 2 modes × 5 seeds)
seeds_arg = ' '.join(str(s) for s in SEEDS)
!python scripts/run_multiseed_ablation.py \
    --seeds {seeds_arg} \
    --output-dir {MULTISEED_OUTPUT_DIR}


[1/10] === dft | seed=0 ===
Training 3DSC MVP:   0%|                                | 0/50 [00:00<?, ?it/s]/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/composition.py:1372: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/python3.12/site-packages/pymatgen/core/composition.py:1372: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/Users/baptistecaillerie/Documents/Crys_JEPA/.venv/lib/py

In [ ]:
# Afficher les résultats : moyenne ± écart-type par mode
import pandas as pd
from IPython.display import display

per_seed = pd.read_csv(PROJECT_DIR / MULTISEED_OUTPUT_DIR / 'multiseed_per_seed.csv')
summary  = pd.read_csv(PROJECT_DIR / MULTISEED_OUTPUT_DIR / 'multiseed_summary.csv')

print('=== Résultats par seed ===')
display(per_seed.sort_values(['input_mode', 'seed']))

print('\n=== Résumé stabilité (moyenne ± écart-type sur 5 seeds) ===')
KEY_METRICS = ['f1', 'roc_auc', 'mae', 'mae_high_tc', 'accuracy']
fmt = summary[['input_mode', 'n_seeds']].copy()
for col in KEY_METRICS:
    mean_c, std_c = f'{col}_mean', f'{col}_std'
    if mean_c in summary.columns:
        fmt[col] = summary.apply(
            lambda r, m=mean_c, s=std_c: f"{r[m]:.4f} ± {r[s]:.4f}" if pd.notna(r[m]) else 'N/A',
            axis=1,
        )
display(fmt)

if len(summary) >= 2:
    d = summary.set_index('input_mode')
    if 'crys_jepa_dft' in d.index and 'dft' in d.index:
        delta_f1  = d.loc['crys_jepa_dft', 'f1_mean']  - d.loc['dft', 'f1_mean']
        delta_mae = d.loc['dft', 'mae_mean'] - d.loc['crys_jepa_dft', 'mae_mean']
        std_f1    = (d.loc['crys_jepa_dft', 'f1_std']**2 + d.loc['dft', 'f1_std']**2)**0.5
        print(f'\nΔF1  (crys_jepa_dft − dft) = {delta_f1:+.4f}  (±{std_f1:.4f} pooled std)')
        print(f'ΔMAE (dft − crys_jepa_dft) = {delta_mae:+.4f}')
        stable_f1 = abs(delta_f1) > 2 * std_f1
        print(f'\n→ Gain F1 significatif (|Δ| > 2σ) : {"OUI ✓" if stable_f1 else "NON — à confirmer"}')

## Notes

- Le modèle `crys_jepa_dft` utilise la fusion tardive : embedding structurel + MLP DFT.
- Si aucun checkpoint Crys-JEPA réel n'est indiqué, l'encodeur structurel reste le placeholder compatible du projet.
- Pour utiliser un vrai checkpoint Crys-JEPA, renseigne `model.crys_jepa_checkpoint` dans la config avant l'entraînement.
- Sur Apple Silicon (MPS), `use_amp` est désactivé car non supporté par PyTorch MPS.